In [43]:
import pandas as pd
import numpy as np
import math as math
import os

## **Loading Clinical Data and Genomic Data**

- Loading Clinical Data and Genomic Data (CNA matrix) from the publication
- Calculating mutational signature data

In [44]:
# clinical data
suppTables = "PublicationData/Supplementary_Tables.xlsx"
suppData = "PublicationData/Supplementary_Data.xlsx"

clinicalSheet = "Supplemental Table 1"

clinicalDF = pd.read_excel(suppTables, sheet_name = clinicalSheet, index_col=0, header=2).iloc[0:144]
clinicalDF[0:5]

,total_muts,nonsyn_muts,clonal_muts,subclonal_muts,heterogeneity,total_neoantigens,CNA_prop,"gender (Male=1, Female=0)",biopsy site,monthsBiopsyPreTx,...,postCTLA4,postMAPKTx,postCombinedCTLA_PD1,numPriorTherapies,biopsy site_categ,biopsyContext (1=Pre-Ipi; 2=On-Ipi; 3=Pre-PD1; 4=On-PD1),daysBiopsyToPD1,daysBiopsyAfterIpiStart,purity,ploidy
Patient1,34.0,22.0,12.0,10.0,0.454545,49.0,0.321417,0.0,skin,2.8,...,0.0,0.0,0.0,1.0,skin,3.0,-84.0,na,0.92,1.73
Patient10,96.0,71.0,48.0,22.0,0.314286,230.0,0.391384,0.0,skin,0.4,...,1.0,1.0,0.0,2.0,skin,3.0,-12.0,107,0.83,1.84
Patient100,200.0,126.0,98.0,24.0,0.196721,301.0,0.029447,0.0,skin,3.1,...,0.0,0.0,0.0,1.0,skin,3.0,-94.0,33,0.11,2.17
Patient102,370.0,246.0,215.0,26.0,0.107884,825.0,0.169389,1.0,brain,2.1,...,0.0,0.0,0.0,0.0,brain,3.0,-64.0,na,0.70,3.24
Patient104,130.0,96.0,65.0,28.0,0.301075,329.0,0.206518,0.0,lymph node,1.9,...,0.0,0.0,0.0,0.0,lymph node,3.0,-57.0,na,0.86,4.58


In [45]:
# mutational signature data
mutsigSheet = "Mutational Sig Activity"

mutsigDF = pd.read_excel(suppData, sheet_name = mutsigSheet, header=3, index_col=0)
print(mutsigDF[0:5])

mutsigDF.columns = ['UV', 'Alkylating', 'Cosmic 1+5']

                     UV    Alkylating  Cosmic 1+5 (Aging+ Signature)
Patient                                                             
Patient145   370.425028  4.268520e+00                      42.307208
Patient125    38.074134  1.037356e+01                      36.549862
Patient100   134.899883  1.269846e+01                      33.400423
Patient62   8941.997050  1.320000e-09                     354.071545
Patient204  2168.898181  9.107963e+01                     292.022232


In [46]:
# merge in clinical and DNA data 
tumorDF = clinicalDF.merge(mutsigDF, left_index=True, right_index=True)
tumorDF[0:5]

,total_muts,nonsyn_muts,clonal_muts,subclonal_muts,heterogeneity,total_neoantigens,CNA_prop,"gender (Male=1, Female=0)",biopsy site,monthsBiopsyPreTx,...,numPriorTherapies,biopsy site_categ,biopsyContext (1=Pre-Ipi; 2=On-Ipi; 3=Pre-PD1; 4=On-PD1),daysBiopsyToPD1,daysBiopsyAfterIpiStart,purity,ploidy,UV,Alkylating,Cosmic 1+5
Patient1,34.0,22.0,12.0,10.0,0.454545,49.0,0.321417,0.0,skin,2.8,...,1.0,skin,3.0,-84.0,na,0.92,1.73,0.509426,2.514556,28.973872
Patient10,96.0,71.0,48.0,22.0,0.314286,230.0,0.391384,0.0,skin,0.4,...,2.0,skin,3.0,-12.0,107,0.83,1.84,35.883346,6.810735,42.303101
Patient100,200.0,126.0,98.0,24.0,0.196721,301.0,0.029447,0.0,skin,3.1,...,1.0,skin,3.0,-94.0,33,0.11,2.17,134.899883,12.698465,33.400423
Patient102,370.0,246.0,215.0,26.0,0.107884,825.0,0.169389,1.0,brain,2.1,...,0.0,brain,3.0,-64.0,na,0.70,3.24,281.460930,24.053939,42.484586
Patient104,130.0,96.0,65.0,28.0,0.301075,329.0,0.206518,0.0,lymph node,1.9,...,0.0,lymph node,3.0,-57.0,na,0.86,4.58,15.669807,3.587685,87.736229


In [47]:
# read in CNA matrix by patient -- for convenience, this has been preprocessed
cnaSheet = "Gene CNAs"
# DEFINE mutation and CNA types
MULTIPLE_MUTS = 3
SILENT = 0
MISSENSE_MUTATION = 1
TRUNCATED_MUTATION = 2
INFRAME_INDEL = 8

AMP = 6
HIGH_AMP = 7
HOMOZYGOUS_DEL = 5
LOH = 4
HOMOZYGOUS_DEL_AND_AMP = 9
cnaDF = pd.read_excel(suppData,sheet_name = cnaSheet, header=4,index_col=0)

def CNAType(cna):
    translateDict = {"HDEL":HOMOZYGOUS_DEL, 
                     "AMP":AMP,
                     "HIGH_AMP":HIGH_AMP,
                     "LOH":LOH,
                     "HDEL_AMP":HOMOZYGOUS_DEL_AND_AMP
                     }
    if cna in translateDict:
        return translateDict[cna]
    else:
        try:
            out = int(float(cna))
            return out
        except: # not a numeric type
            return cna
    
def CNAType_Series(cnaSeries):
    return(cnaSeries.apply(CNAType))

cnaDF = cnaDF.apply(CNAType_Series)[tumorDF.index] # limit to the 144 tumors in our cohort

cnaDF[0:5]

,Patient1,Patient10,Patient100,Patient102,Patient104,Patient105,Patient106,Patient107,Patient108,Patient11,...,Patient83,Patient84,Patient86,Patient87,Patient88,Patient9,Patient94,Patient96,Patient98,Patient99
Gene,,,,,,,,,,,,,,,,,,,,,
5S_rRNA,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,4,4,4,0,0,4
7SK,0,4,0,0,0,4,0,0,0,0,...,0,0,0,0,0,0,4,0,0,0
A1BG,0,0,0,0,0,4,0,0,0,0,...,0,0,0,0,0,0,4,0,0,4
A1BG-AS1,0,0,0,0,0,4,0,0,0,0,...,0,0,0,0,0,0,4,0,0,4
A1CF,4,4,0,4,4,4,4,4,0,4,...,0,0,0,0,4,0,4,4,4,0


## **Loading Transcriptomics Data**

In [48]:
# read in RNA data
allRNAFile = "PublicationData/GeneExpression.txt"
allRNADF = pd.read_csv(allRNAFile, sep="\t", low_memory=False,index_col=0)

## **Raw Processing Data**

**Clinical and Genomic Data**
- Generating amplified tumors for HLA MHC-I genes and TAP2
- Standardizing Arm and IOTherapy
- Standardizing "daysBiopsyAfterIpiStart" into 3 values: noIpi; PreIpi; PostIpi
- Droping uninteresting columns
- Refining column names

In [49]:
# generate amplified tumors for HLA MHC-I genes and TAP2
ampTumors= {}

ampGenes = ['HLA-A', 'HLA-B', 'HLA-C', 'TAP2']

for gene in ampGenes:
    tmp = cnaDF.loc[gene].isin([AMP, HIGH_AMP])
    ampTumors[gene] = list(tmp[tmp].index)

# add additional amp genes
ampTumors['MHC-I HLA'] = set(ampTumors['HLA-A']).intersection(\
                     set(ampTumors['HLA-B']).intersection(\
                     set(ampTumors['HLA-C'])))
ampTumors['MHC-I HLA']

# Create the new columns and initialize to 0
tumorDF['MHC-I amp'] = 0
tumorDF['TAP2 amp'] = 0

# --- Fix is here: Convert the set to a list using list() ---
# Use the list of tumors to set the 'MHC-I amp' value to 1
tumorDF.loc[list(ampTumors['MHC-I HLA']), 'MHC-I amp'] = 1

# Use the list of tumors to set the 'TAP2 amp' value to 1
tumorDF.loc[list(ampTumors['TAP2']), 'TAP2 amp'] = 1

In [50]:
# 3.remove spaces and lowcases
# Dùng .astype(str) to handle None/NaN nếu có
tumorDF['IOTherapy'] = (
    tumorDF['IOTherapy']
    .astype(str)
    .str.strip()
    .str.lower()
)

# 4. Standardizing Arms
standardization_map = {
    # Pembrolizumab
    'pembro': 'Pembrolizumab',
    'pembrolizumab': 'Pembrolizumab',
    'mk3475': 'Pembrolizumab',
    'mk 3475': 'Pembrolizumab',

    # Nivolumab
    'nivo': 'Nivolumab',
    'nivolumab': 'Nivolumab'
}

tumorDF['IOTherapy'] = tumorDF['IOTherapy'].replace(standardization_map)

In [51]:
# Standardizing "daysBiopsyAfterIpiStart" into noIpi, preIpi, PostIPi
Ipicols = "daysBiopsyAfterIpiStart"
ipiTreated = pd.to_numeric(tumorDF['daysBiopsyAfterIpiStart'], errors = 'coerce')
postIpi = list(ipiTreated.loc[ipiTreated>0].index)
preIpi = list(ipiTreated.loc[ipiTreated<=0].index)
noIpi = list(tumorDF.loc[tumorDF[Ipicols]=='na'].index)

tumorDF.loc[postIpi, Ipicols]="postIpi"
tumorDF.loc[preIpi, Ipicols]="preIpi"
tumorDF.loc[noIpi, Ipicols]="noIpi"

In [52]:
tumorDF = tumorDF.drop(columns = ["Histology","biopsy site","Tx"])

In [53]:
tumorDF = tumorDF.rename(columns = {"gender (Male=1, Female=0)": "gender",
                                'Mstage (IIIC=0, M1a=1, M1b=2, M1c=3)': "Mstage",
                                'biopsy site_categ':  'biopsy_site_categ',
                                'biopsyContext (1=Pre-Ipi; 2=On-Ipi; 3=Pre-PD1; 4=On-PD1)':"biopsyContext",
                                'Cosmic 1+5': "Cosmic",
                                'MHC-I amp': "MHCI_Amp",
                                "TAP2 amp": "TAP2_Amp"
})

**Transcriptomics Data**

- Removing genes whose zero expression proportion exceeds 50%.

In [54]:
genelist = allRNADF.columns.tolist()
removegenelist = []
for gene in genelist:
    counts_0 = allRNADF[gene].values.tolist().count(0)
    proportion = counts_0/(allRNADF.shape[0])
    if proportion > 0.5:
        removegenelist.append(gene)

print(f"There are {len(removegenelist)} removed genes.")
print(f"Initial Shape {allRNADF.shape}")
cleaned_RNADF = allRNADF.drop(columns = removegenelist)        
print(f"Cleaned Shape {cleaned_RNADF.shape}")

There are 2088 removed genes.
Initial Shape (121, 20848)
Cleaned Shape (121, 18760)


## **Saving Data**

In [55]:
savefolder = "MelanomaData"
os.makedirs(savefolder, exist_ok=True)
tumorDF.to_csv(f"{savefolder}/clinical_data.csv", index = True)
cleaned_RNADF.to_csv(f"{savefolder}/TranscriptomicsData.csv", index = True)